# Sweep — `feature/mask-rcnn` postprocess на `best.pt`

Не учим. Коммит `c949b21`: быстрый PQ + сетка score/min_area/dedup.

GPU T4, Internet on.
Add Data: competition + Output `dm1triyaleksandrov/notebookead6e030d5`.
Add-ons → Secrets: `GITHUB_TOKEN` и `DAGSHUB_TOKEN` на **этот** ноутбук.

После sweep — полный val лучшим конфигом и CSV, если PQ ≥ 0.20. Результат пишется в DagsHub (MLflow).

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Session options → GPU T4, Restart session"

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import os

token = ""
dagshub_token = ""
try:
    sec = UserSecretsClient()
    for name in ("GITHUB_TOKEN", "GH_TOKEN", "GIT_TOKEN", "github_token"):
        try:
            value = sec.get_secret(name)
        except Exception:
            value = ""
        if value:
            token = value
            break
    try:
        dagshub_token = sec.get_secret("DAGSHUB_TOKEN") or ""
    except Exception:
        dagshub_token = ""
except Exception as e:
    print("secrets:", type(e).__name__, e)

os.environ["GITHUB_TOKEN"] = token
if dagshub_token:
    os.environ["DAGSHUB_TOKEN"] = dagshub_token
if not token:
    raise SystemExit("Add-ons → Secrets → GITHUB_TOKEN на ЭТОТ ноутбук")
if not dagshub_token:
    raise SystemExit("Add-ons → Secrets → DAGSHUB_TOKEN на ЭТОТ ноутбук")

askpass = """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""
Path("/kaggle/working/git-askpass.sh").write_text(askpass)
os.chmod("/kaggle/working/git-askpass.sh", 0o700)

REPO = Path("/kaggle/working/repo")
if not (REPO / "run.py").exists():
    !GIT_ASKPASS=/kaggle/working/git-askpass.sh GIT_TERMINAL_PROMPT=0 \
      git clone --branch feature/mask-rcnn --single-branch \
      https://github.com/d1maal/filament-segmentation-2026.git /kaggle/working/repo
else:
    !GIT_ASKPASS=/kaggle/working/git-askpass.sh GIT_TERMINAL_PROMPT=0 \
      git -C /kaggle/working/repo fetch origin feature/mask-rcnn
    !git -C /kaggle/working/repo checkout -B feature/mask-rcnn origin/feature/mask-rcnn

%cd /kaggle/working/repo
!git log -1 --oneline
assert Path("src/models/mask_rcnn/sweep_postprocess.py").exists(), "нужен коммит c949b21+"

env_lines = Path(".env.example").read_text().splitlines()
out_lines = []
for line in env_lines:
    if line.startswith("DAGSHUB_TOKEN="):
        out_lines.append(f"DAGSHUB_TOKEN={dagshub_token}")
    elif line.startswith("DAGSHUB_EXPERIMENT_NAME="):
        out_lines.append("DAGSHUB_EXPERIMENT_NAME=mask_rcnn_postprocess")
    else:
        out_lines.append(line)
Path(".env").write_text("\n".join(out_lines) + "\n")
os.environ["DAGSHUB_EXPERIMENT_NAME"] = "mask_rcnn_postprocess"
os.environ["DAGSHUB_REPO_OWNER"] = "BraveCactus"
os.environ["DAGSHUB_REPO_NAME"] = "solar-filament-segmentation"
print("DagsHub .env ready → https://dagshub.com/BraveCactus/solar-filament-segmentation")

%pip install -q pycocotools python-dotenv tqdm mlflow dagshub matplotlib

**DAGSHUB / MLflow** — эта ячейка создаёт прогон. Секрет `DAGSHUB_TOKEN` должен быть прикреплён.

In [ ]:
import dagshub
import mlflow

dagshub.init(
    repo_owner="BraveCactus",
    repo_name="solar-filament-segmentation",
    mlflow=True,
)
mlflow.set_experiment("mask_rcnn_postprocess")
run = mlflow.start_run(run_name="mask-rcnn-postprocess-sweep")
mlflow.log_param("model", "mask_rcnn")
mlflow.log_param("branch", "feature/mask-rcnn")
print("run_id:", run.info.run_id)
print("experiment:", mlflow.get_experiment(run.info.experiment_id).name)

**SWEEP** — один forward на кадр, ~30 конфигов. Не `src.eval` (там mlflow).

In [ ]:
from pathlib import Path

import mlflow

from src.models.mask_rcnn.sweep_postprocess import run_sweep

WEIGHTS = Path("/kaggle/working/outputs/mask_rcnn/best.pt")
if not WEIGHTS.exists():
    hits = sorted(Path("/kaggle/input").rglob("best.pt"))
    assert hits, "Add Input → Output прогона notebookead6e030d5"
    WEIGHTS = hits[0]

OUT = Path("/kaggle/working/outputs/mask_rcnn")
OUT.mkdir(parents=True, exist_ok=True)
print("weights:", WEIGHTS)
mlflow.log_param("weights", str(WEIGHTS))

results = run_sweep(
    WEIGHTS,
    limit=0,
    out_csv=OUT / "postprocess_sweep.csv",
)
best_cfg = results[0]
print("BEST CFG:", best_cfg)
mlflow.log_params({f"best_{k}": v for k, v in best_cfg.items() if k != "pq"})
mlflow.log_metrics({"sweep_best_pq": float(best_cfg["pq"])})
mlflow.log_artifact(str(OUT / "postprocess_sweep.csv"))

**FULL VAL + CSV** лучшим конфигом. Сабмит только если PQ ≥ 0.20.

In [ ]:
import csv
import json
from pathlib import Path

import cv2
import mlflow
import numpy as np
import torch
from tqdm import tqdm

from src.config import config
from src.data import (
    ensure_val_split,
    image_stem,
    instances_for_image,
    load_coco,
    mask_to_rle_counts,
    resolve_image,
    three_channel,
)
from src.metrics import aggregate, panoptic_quality
from src.models.mask_rcnn.infer import load_checkpoint, predict_masks

MIN_PQ_FOR_SUBMIT = 0.20
OUT = Path("/kaggle/working/outputs/mask_rcnn")
OUT.mkdir(parents=True, exist_ok=True)


def val_items():
    coco = load_coco()
    val = ensure_val_split(coco)
    items = []
    for info in coco.imgs.values():
        if image_stem(info["file_name"]) not in val:
            continue
        path = resolve_image(config.train_images, info["file_name"])
        if path is not None:
            items.append((coco, info, path))
    items.sort(key=lambda item: image_stem(item[1]["file_name"]))
    return items


def to_tensor(gray):
    rgb = three_channel(gray)
    return torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))).float() / 255.0


model, device = load_checkpoint(WEIGHTS)
items = val_items()
rows = []
print(
    f"full val {len(items)} | score={best_cfg['score_threshold']} "
    f"area={best_cfg['min_area']} dedup={best_cfg['deduplicate']}",
    flush=True,
)
for coco, info, path in tqdm(items, desc="val"):
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise SystemExit(f"Failed to read {path}")
    pred = predict_masks(
        model,
        to_tensor(gray),
        device,
        score_threshold=best_cfg["score_threshold"],
        min_area=best_cfg["min_area"],
        deduplicate=best_cfg["deduplicate"],
        mask_nms_iou=best_cfg["mask_nms_iou"],
        containment_threshold=best_cfg["containment_threshold"],
    )
    metrics = panoptic_quality(instances_for_image(coco, info["id"]), pred)
    metrics["stem"] = Path(info["file_name"]).stem
    rows.append(metrics)

summary = aggregate(rows)
print("Home PQ full val:", round(summary["pq"], 4), summary)
(OUT / "val_metrics.json").write_text(
    json.dumps({"weights": str(WEIGHTS), "cfg": best_cfg, "summary": summary, "per_image": rows}, indent=2),
    encoding="utf-8",
)
mlflow.log_metrics(
    {
        "val_pq": float(summary["pq"]),
        "val_tp": int(summary.get("tp", 0)),
        "val_fp": int(summary.get("fp", 0)),
        "val_fn": int(summary.get("fn", 0)),
        "val_splits": int(summary.get("splits", 0)),
        "val_merges": int(summary.get("merges", 0)),
        "val_images": int(summary.get("images", 0)),
    }
)
mlflow.log_artifact(str(OUT / "val_metrics.json"))

SUBMIT = summary["pq"] >= MIN_PQ_FOR_SUBMIT
print("submit:", SUBMIT, f"(threshold {MIN_PQ_FOR_SUBMIT})")
if SUBMIT:
    csv_rows = []
    test_images = sorted(
        p for p in config.test_images.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    print(f"test: {len(test_images)}", flush=True)
    for path in tqdm(test_images, desc="test"):
        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            raise SystemExit(f"Failed to read {path}")
        pred = predict_masks(
            model,
            to_tensor(gray),
            device,
            score_threshold=best_cfg["score_threshold"],
            min_area=best_cfg["min_area"],
            deduplicate=best_cfg["deduplicate"],
            mask_nms_iou=best_cfg["mask_nms_iou"],
            containment_threshold=best_cfg["containment_threshold"],
        )
        for i, mask in enumerate(pred, start=1):
            csv_rows.append(
                {"filament_id": f"{path.stem}_{i}", "segmentation_rle": mask_to_rle_counts(mask)}
            )
    sub = OUT / "submission_mask_rcnn.csv"
    with sub.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, ["filament_id", "segmentation_rle"])
        w.writeheader()
        w.writerows(csv_rows)
    print(f"Wrote {sub} ({len(csv_rows)} rows)")
    mlflow.log_artifact(str(sub))
    !ls -lh /kaggle/working/outputs/mask_rcnn/

mlflow.end_run()
print("DagsHub run closed")